[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohsennasab/Rang/blob/main/tools/examole/06_build_palette.ipynb)

<div style="font-family:Arial,sans-serif">

# 06. Build the palette

Write the palette JSON, run the repository build, and inspect the generated page.

Created by **Mohsen Tahmasebi Nasab, PhD**<br>
[hydromohsen.com](https://hydromohsen.com)

Copyright and license holder: Mohsen Tahmasebi Nasab. Notebook code is
licensed under the repository's MIT License. Rang palette data follows the
CC0 dedication described in the licensing guide. Source images keep their own
rights and reuse terms.

</div>

<div style="font-family:Arial,sans-serif;background:#fff3cd;padding:14px"><strong>YOUR INPUT</strong><br>Set the Git branch, choose whether to use Google Drive, and give the palette a short filename.</div>

In [ ]:
REPO_REF = "main" #@param {type:"string"}
USE_GOOGLE_DRIVE = True #@param {type:"boolean"}
PALETTE_SLUG = "kashan" #@param {type:"string"}

Google Drive keeps the recipe available when you move to the next notebook.
For a local Jupyter session, working files are placed under
`cache/notebook_workflow/`. When testing a GitHub branch, replace `main` with
the branch name above.

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = False
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    REPO_ROOT = Path("/content/Rang")
    if not (REPO_ROOT / "tools" / "notebook_workflow.py").exists():
        subprocess.run([
            "git", "clone", "--depth", "1", "--branch", REPO_REF,
            "https://github.com/mohsennasab/Rang.git", str(REPO_ROOT)
        ], check=True)
else:
    probe = Path.cwd().resolve()
    REPO_ROOT = next(
        candidate for candidate in (probe, *probe.parents)
        if (candidate / "tools" / "notebook_workflow.py").exists()
    )

try:
    import matplotlib
    import numpy
    import PIL
    import sklearn
except ImportError:
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q", "-r",
        str(REPO_ROOT / "tools" / "requirements.txt")
    ], check=True)

sys.path.insert(0, str(REPO_ROOT / "tools"))

from notebook_workflow import *

if IN_COLAB and USE_GOOGLE_DRIVE:
    drive.mount("/content/drive")
    WORK_DIR = Path("/content/drive/MyDrive/Rang") / PALETTE_SLUG
else:
    WORK_DIR = REPO_ROOT / "cache" / "notebook_workflow" / PALETTE_SLUG

WORK_DIR.mkdir(parents=True, exist_ok=True)
RECIPE_PATH = WORK_DIR / f"{PALETTE_SLUG}-recipe.json"
use_arial()
print("Repository:", REPO_ROOT)
print("Working folder:", WORK_DIR)
print("Recipe:", RECIPE_PATH)

In [ ]:
if not RECIPE_PATH.exists():
    raise FileNotFoundError(
        f"Recipe not found at {RECIPE_PATH}. Run notebook 01 first and use "
        "the same Google Drive and palette slug settings."
    )
recipe = read_json(RECIPE_PATH)
print(f'Loaded {recipe["palette"]} with {len(recipe["regions"])} regions')

<div style="font-family:Arial,sans-serif;background:#fff3cd;padding:14px"><strong>YOUR INPUT</strong><br>Enter the artwork metadata exactly as the museum or source page gives it. Set the rights fields carefully.</div>

In [ ]:
PALETTE_METADATA = {
    "name": "Kashan",
    "persian": "کاشان",
    "pronunciation": "kah-SHAHN",
    "position": 1,
    "source": {
        "title": "Silk Kashan Carpet",
        "artist": "",
        "date": "16th century",
        "geography": "Made in Iran, probably Kashan",
        "medium": "Silk (warp, weft and pile), asymmetrically knotted pile",
        "museum": "The Metropolitan Museum of Art, New York",
        "department": "Islamic Art",
        "accession": "58.46",
        "credit": "Gift of Mrs. Douglas M. Moffat, 1958",
        "url": "https://www.metmuseum.org/art/collection/search/451470",
        "image": "https://images.metmuseum.org/CRDImages/is/original/DT5450.jpg",
        "card_image": "sources/kashan/card.jpg",
        "public_domain": True
    }
}

<div style="font-family:Arial,sans-serif;background:#d9edf7;padding:14px"><strong>YOUR DECISION</strong><br>Build inside the Colab clone after the draft is written. Leave this off until the metadata and rights are ready.</div>

In [ ]:
RUN_BUILD = True #@param {type:"boolean"}

In [ ]:
draft_path = WORK_DIR / f"{PALETTE_SLUG}-palette.json"
draft = palette_draft(RECIPE_PATH, PALETTE_METADATA, draft_path)
print("Palette draft:", draft_path)
print("Final colors:", *draft["colors"])

if RUN_BUILD:
    repository_palette = REPO_ROOT / "palettes" / f"{PALETTE_SLUG}.json"
    shutil.copyfile(draft_path, repository_palette)
    subprocess.run([
        sys.executable, str(REPO_ROOT / "tools" / "build.py"), PALETTE_SLUG
    ], cwd=REPO_ROOT, check=True)
    print("Generated page:", REPO_ROOT / "docs" / PALETTE_SLUG / "README.md")

The build inside Colab is a validation copy. Download the recipe and palette
draft, then add them to your own Git branch with the generated repository
files. Check the artwork preview and every sample plot before opening a pull
request.